In [ ]:
# Import adata and load the rna data 
import anndata as ad

DATA_DIR = "/home/ubuntu/data/frangieh"

rna_data_original = ad.read_h5ad(f"{DATA_DIR}/rna.h5ad")

In [ ]:
# Load the protein data
protein_data_original = ad.read_h5ad(f"{DATA_DIR}/protein.h5ad")

In [ ]:
import scanpy as sc

# Flag mitochondrial genes (human naming convention; adjust prefix if needed)
rna_data_original.var["mt"] = rna_data_original.var_names.str.upper().str.startswith("MT-")

sc.pp.calculate_qc_metrics(
    rna_data_original, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

In [ ]:
# Inspect QC metric distributions before choosing filtering thresholds
sc.pl.violin(
    rna_data_original,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# Doublet detection (not covered by the existing precomputed QC columns)
sc.pp.scrublet(rna_data_original)

rna_data_original.obs["predicted_doublet"].value_counts()

In [ ]:
# Conservative filtering: this data has already been QC'd upstream (see notebook discussion),
# so we only trim a floor + a mito ceiling and drop predicted doublets, rather than
# re-tightening thresholds around the bulk of the existing distribution.
# Built as a single combined mask + one copy to avoid stacking up multiple
# full-size intermediate copies in memory alongside rna_data_original.
keep = (
    ~rna_data_original.obs["predicted_doublet"]
    & (rna_data_original.obs["n_genes_by_counts"] >= 200)
    & (rna_data_original.obs["total_counts"] >= 500)
    & (rna_data_original.obs["pct_counts_mt"] < 18)
)
rna_data = rna_data_original[keep].copy()

sc.pp.filter_genes(rna_data, min_cells=3)

print(f"Cells: {rna_data_original.n_obs} -> {rna_data.n_obs}")
print(f"Genes: {rna_data_original.n_vars} -> {rna_data.n_vars}")

In [ ]:
# Per-perturbation cell count check: make sure filtering hasn't wiped out
# any single perturbation's representation
before = rna_data_original.obs["perturbation"].value_counts()
after = rna_data.obs["perturbation"].value_counts()

perturbation_counts = (
    before.to_frame("n_cells_before")
    .join(after.to_frame("n_cells_after"))
    .fillna(0)
)
perturbation_counts["retained_frac"] = (
    perturbation_counts["n_cells_after"] / perturbation_counts["n_cells_before"]
)

perturbation_counts.sort_values("retained_frac").head(20)